# Sanskrit TTS Generator
Run this first cell to load the model and tokenizer. This will take a few seconds but only needs to happen once.

## Setup

Install dependencies once from the project root (PowerShell):

`powershell
python -m venv tts
.\tts\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook sanskrit_tts.ipynb
`

Then select the tts kernel and run the cells below. See README.md for full details.


In [2]:
import torch
import soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL = "ai4bharat/indic-parler-tts"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device}...")
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
print("Model loaded successfully!")

Loading model on cpu...


Config of the text_encoder: <class 'transformers.models.t5.modeling_t5.T5EncoderModel'> is overwritten by shared text_encoder config: T5Config {
  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

Config of the audio_encoder: <class 'transformers.models.dac.modelin

Model loaded successfully!


### Define Generation Function
Run this cell to define the reusable function.

In [6]:
def generate_sanskrit_audio(text, output_filename="sanskrit_output.mp3", description=None):
    if description is None:
        description = """
            Aryan is a trained Sanskrit male reciter performing a traditional Vedic hymn.
            He has a deep, clear, warm and reverent male voice with precise Sanskrit pronunciation.
            He articulates every Sanskrit syllable distinctly and naturally, without swallowing or
            merging consonants. The recitation is steady, measured, solemn and devotional.
            He follows the natural rhythm and meter of the Sanskrit hymn, maintaining consistent
            timing and breath control. Each pāda is clearly articulated with a gentle, natural pause
            at its boundary, and there is a slightly longer pause between complete verses.
            The voice is focused and meditative, with restrained expression and no theatrical acting.
            The recitation sounds like a disciplined traditional Sanskrit pāṭha rather than ordinary
            spoken language. Maintain consistent pronunciation, volume, pitch and tempo throughout.
            Use clean studio-quality audio with a dry, natural acoustic environment and no background
            music, instruments, echo, reverb, or sound effects.
        """
    
    description_inputs = description_tokenizer(description, return_tensors="pt").to(device)
    text_inputs = tokenizer(text, return_tensors="pt").to(device)

    print(f"Generating audio...")
    generation = model.generate(
        input_ids=description_inputs.input_ids,
        attention_mask=description_inputs.attention_mask,
        prompt_input_ids=text_inputs.input_ids,
        prompt_attention_mask=text_inputs.attention_mask,
    )

    audio = generation.cpu().numpy().squeeze()
    sf.write(output_filename, audio, model.config.sampling_rate)
    print(f"Saved: {output_filename}")
    
    from IPython.display import Audio, display
    display(Audio(output_filename, autoplay=False))

### Generate Audio for Ganesha Sloka

In [7]:
ganesha_sloka = """
प्रणम्य शिरसा देवं गौरीपुत्रं विनायकम् ।
भक्तावासं स्मरेन्नित्यमायुःकामार्थसिद्धये ॥
"""
generate_sanskrit_audio(ganesha_sloka, "ganesha_sloka.mp3")

Generating audio...
Saved: ganesha_sloka.mp3


### Generate Audio for Shanti Mantra

In [5]:
shanti_mantra = """
ॐ सह नाववतु ।
सह नौ भुनक्तु ।
सह वीर्यं करवावहै ॥
"""
generate_sanskrit_audio(shanti_mantra, "shanti_mantra.mp3")

Generating audio...
Saved: shanti_mantra.mp3
